- 001 hello world
- 002 save only max value of samegrant/sameagency

In [ ]:
from ppp_paths import databulk, dataint, dataconf


In [ ]:
import pandas as pd
import unicodedata
import re
import gc

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')
possiblepairs['doi'] = possiblepairs['doi'].str.upper()
possiblepairs

In [ ]:
import unicodedata
import re

def normalizegrant(data):
    normal = unicodedata.normalize('NFKD', data).encode('ASCII', 'ignore')
    val = normal.decode("utf-8")
    # Lower case
    val = val.lower()
    # remove special characters
    val = re.sub('[^A-Za-z0-9 ]+', ' ', val)
    # remove  spaces
    val = re.sub(' *', '', val)
    # remove trailing spaces
    val = val.strip()
    return val
normalizegrant("f1-355ab")

In [ ]:
govtext = pd.read_csv(databulk + "g_gov_interest.tsv.zip", sep='\t', compression='zip', dtype=str)
govcontracts = pd.read_csv(databulk + "g_gov_interest_contracts.tsv.zip", sep='\t', compression='zip', dtype=str)
govcontracts['contract_award_number'] = govcontracts['contract_award_number'].apply(normalizegrant)
govorgs = pd.read_csv(databulk + "g_gov_interest_org.tsv.zip", sep='\t', compression='zip', dtype=str)
govcontracts    


In [ ]:
wosdoi = pd.read_stata(dataconf + 'wosdoi.dta')
wosdoi['doi'] = wosdoi['doi'].str.upper()
wosdoi

In [ ]:
print(possiblepairs.shape) 
possiblepairs = pd.merge(possiblepairs, wosdoi, on='doi', how='left')
print(possiblepairs.shape)

In [ ]:
wosgrantnums = pd.read_stata(dataconf + 'wosgrantnums.dta')
wosgrantnums['grantnum'] = wosgrantnums['grantnum'].apply(normalizegrant)
wosgrantnums

In [ ]:
wosgrantagencies = pd.read_stata(dataconf + 'wosusgrants.dta')
wosgrantagencies = wosgrantagencies.rename(columns={'usgrantagency':'papergrantagency'})
wosgrantagencies

In [ ]:
possiblepairs = pd.merge(possiblepairs, wosgrantagencies, on='wosid', how='left')
possiblepairs = pd.merge(possiblepairs, wosgrantnums, on='wosid', how='left')
possiblepairs = pd.merge(possiblepairs, govtext, on='patent_id', how='left')
possiblepairs = pd.merge(possiblepairs, govorgs, on='patent_id', how='left')
possiblepairs = pd.merge(possiblepairs, govcontracts, on='patent_id', how='left')
possiblepairs

In [ ]:
possiblepairs = possiblepairs[['magid', 'patent_id',  'papergrantagency', 'grantnum', 'fedagency_name', 'level_one', 'level_two', 'level_three', 'contract_award_number', 'gi_statement']]
possiblepairs

In [ ]:
possiblepairsna = possiblepairs.copy()
# drop if no grant info on paper side
possiblepairsna = possiblepairsna.dropna(subset=['papergrantagency', 'grantnum'], how='all')
# drop if no grant info on patent side
possiblepairsna = possiblepairsna.dropna(subset=['fedagency_name', 'contract_award_number'], how='all')
possiblepairsna

In [ ]:
import re

def map_agency(s):
    p = None
    s = s.lower()
    if pd.isna(s):
        return ""

    # Department of Defense (DOD)
    if re.search(r'us navy|naval observatory|army usa|army|navy usa|air force usa|united states navy|the navy|the army|the air force|secretary of navy|secretary of army|secretary of air|secretary of the navy|secretary of the army|secretary of the air|s marine corp|us army|united states army|us air force|united states air force|air force|usaf|darpa|defense advance|t of defense|y of defense|missile|^dod$|coast guard|national security|naval|naval res|naval academy', s):
        p = "DOD"
    
    # Department of Veterans Affairs (USDVA)
    elif re.search(r'department of veterans affairs', s):
        p = "USDVA"
    
    # National Science Foundation (NSF)
    elif re.search(r'natl sci fdn|national science foundation|n ?s ?f(?!nnsfc)', s):
        p = "NSF"
    
    # Department of Energy (DOE)
    elif re.search(r't of energy|d ?o ?e|^nist$|ioffice of science of the u\.s|us atomic energy|atomic energy commission|y of energy|united states energy', s, re.IGNORECASE):
        p = "DOE"
    
    # DOE Laboratories
    elif (re.search(r'natl|national', s) and 
          re.search(r'laborato|lab', s) and 
          re.search(r'ames|argonne|brookhaven|fermi|lawrence|berkeley|oak ridge|northwest|princeton|slac|accelerator|thomas|jefferson|livermore|los alamos|sandia', s)):
        p = "DOE"
    
    # Health and Human Services (HHS) and National Institutes of Health (NIH)
    elif re.search(r'hhs|human.*health svr|human.*health service|blood inst|medicare', s, re.IGNORECASE):
        p = "HHS"
    elif re.search(r'national institutes? of health|natl institutes of health|natl inst hlth|n ?i ?h', s, re.IGNORECASE):
        p = "NIH"
    elif re.search(r'ncats|niaaa|niamsd|national institute of arthritis|national center for advancing translational sciences|nhgri|nibib|national institute of biomedical imaging and bioengineering|national human genome research institute|ninds|niehs|nidcd|niaid|niddk|national institute on alcohol abuse and alcoholism|nhlbi|national institute of dental|national institute of environmental health sciences|nichd|nidcr|national institute on aging|national eye institute|national institute of child health and human development|nati?o?n?a?l insti?t?u?t?e o?f? mental health|nati?o?n?a?l insti?t?u?t?e o?f? allergy|nati?o?n?a?l insti?t?u?t?e o?f? diabetes|nati?o?n?a?l insti?t?u?t?e o?n? aging|nati?o?n?a?l insti?t?u?t?e o?[nf]? drug abuse|national institute of general medical sciences|national institute of neurological disorders and stroke|national cancer inst', s):
        p = "NIH"
    elif re.search(r'natl cancer inst', s):
        p = "S"
    elif re.search(r'nidcr', s):
        p = "NIHS"
    
    # Centers for Disease Control and Prevention (CDC)
    elif re.search(r'cdc|disease control|disease ctrl', s):
        p = "CDC"
    
    # Food and Drug Administration (FDA)
    elif re.search(r'fda|food.*drug', s):
        p = "FDA"
    
    # National Oceanic and Atmospheric Administration (NOAA)
    elif re.search(r'n ?o ?a? a|national oceanic and atmospheric administration', s):
        p = "NOAA"
    
    # American Diabetes Association (ADA)
    elif re.search(r'american diabetes association', s):
        p = "ADA"
    
    # National Aeronautics and Space Administration (NASA)
    elif re.search(r'aeronautic.*space|n ?a ?s ?a', s):
        p = "NASA"
    
    # Department of Education (USED)
    elif re.search(r't of education', s):
        p = "USED"
    
    # Administration for Community Living (ACL)
    elif re.search(r'community living', s):
        p = "ACL"
    
    # Agency for Healthcare Research and Quality (AHRQ)
    elif re.search(r'healthcare research.*quality', s):
        p = "AHRQ"
    
    # Assistant Secretary for Preparedness and Response (ASPR)
    elif re.search(r'preparedness.*response', s):
        p = "ASPR"
    
    # Department of Agriculture (USDA)
    elif re.search(r't of agriculture|agriculture usa|forest service|natural resources conservation|agricultural research service|food and agriculture|animal and plant|rural development|usda', s):
        p = "USDA"
    
    # Environmental Protection Agency (EPA)
    elif re.search(r'environmenta?l? protect', s):
        p = "EPA"
    
    # Smithsonian Institution (SMITH)
    elif re.search(r'smithsonian', s):
        p = "SMITH"
    
    # U.S. Agency for International Development (USAID)
    elif re.search(r'u\W?s\Wagency.*international development', s):
        p = "USAID"
    
    # U.S. Geological Survey (USGS)
    elif re.search(r'u\W?s\Wgeological.*survey', s):
        p = "USGS"
    
    # Department of Homeland Security (DHS)
    elif re.search(r'homeland security', s):
        p = "DHS"
    
    # Office of the Director of National Intelligence (ODNI)
    elif re.search(r'director.*national.*intelligence', s):
        p = "ODNI"
    
    # Department of Transportation (DOT)
    elif re.search(r't of transportation', s):
        p = "DOT"
    
    # Other Agencies
    elif re.search(r'postal service|national security agency|fbi|central intelligence|federal bureau|oceanic and admospheric|international trade comm|federal.*admin|federal.*agency|security and exchange.*exchange|t of commerce', s, re.IGNORECASE):
        p = "OTHERS"
    
    return p

In [ ]:
possiblepairsna['fedagency_name'] = possiblepairsna['fedagency_name'].astype(str)
possiblepairsna['level_one'] = possiblepairsna['level_one'].astype(str)
possiblepairsna['level_two'] = possiblepairsna['level_two'].astype(str)
possiblepairsna['level_three'] = possiblepairsna['level_three'].astype(str)

possiblepairsna['patgrantagency1'] = possiblepairsna['fedagency_name'].apply(map_agency)
possiblepairsna['patgrantagency2'] = possiblepairsna['level_one'].apply(map_agency)
possiblepairsna['patgrantagency3'] = possiblepairsna['level_two'].apply(map_agency)
possiblepairsna['patgrantagency4'] = possiblepairsna['level_three'].apply(map_agency)

# cols = ['fedagency_name', 'level_one', 'level_two', 'level_three']
# for i, col in enumerate(cols, 1):
#     possiblepairsna[f'patgrantagency{i}'] = possiblepairsna[col].map(map_agency)
# possiblepairsna

# def map_agency_row(row):
#     return pd.Series([
#         map_agency(row['fedagency_name']),
#         map_agency(row['level_one']),
#         map_agency(row['level_two']),
#         map_agency(row['level_three'])
#     ])

# possiblepairsna[['patgrantagency1', 'patgrantagency2', 'patgrantagency3', 'patgrantagency4']] = (
#     possiblepairsna.apply(map_agency_row, axis=1)
# )

In [ ]:
possiblepairsna['sameagency'] = (possiblepairsna['papergrantagency'] == possiblepairsna['patgrantagency1']) | (possiblepairsna['papergrantagency'] == possiblepairsna['patgrantagency2']) | (possiblepairsna['papergrantagency'] == possiblepairsna['patgrantagency3']) | (possiblepairsna['papergrantagency'] == possiblepairsna['patgrantagency4'])
print(possiblepairsna.value_counts('sameagency'))
possiblepairsna
possiblepairsna['samegrant'] = possiblepairsna['grantnum'] == possiblepairsna['contract_award_number']
print(possiblepairsna.value_counts('samegrant'))
possiblepairsna

In [ ]:
possiblepairsna = possiblepairsna[['magid', 'patent_id', 'sameagency', 'samegrant']]
possiblepairsna = possiblepairsna.dropna()
possiblepairsna = possiblepairsna.drop_duplicates()
gc.collect()
possiblepairsna

In [ ]:
possiblepairsna

In [ ]:
possiblepairs = possiblepairs[['magid', 'patent_id']]
possiblepairs = possiblepairs.drop_duplicates()
possiblepairs2export = pd.merge(possiblepairs, possiblepairsna, on=['magid', 'patent_id'], how='left')
possiblepairs2export

In [ ]:
print(possiblepairs2export['sameagency'].isna().sum())
print(possiblepairs2export['samegrant'].isna().sum())
possiblepairs2export['sameagency'] = possiblepairs2export['sameagency'].fillna(0)
possiblepairs2export['samegrant'] = possiblepairs2export['samegrant'].fillna(0)
print(possiblepairs2export['sameagency'].isna().sum())
print(possiblepairs2export['samegrant'].isna().sum())
possiblepairs2export

In [ ]:
possiblepairs2export['sameagency'] = possiblepairs2export['sameagency'].astype(int)
possiblepairs2export['samegrant'] = possiblepairs2export['samegrant'].astype(int)

In [ ]:
possiblepairs2export.value_counts(['sameagency', 'samegrant'])

In [ ]:
possiblepairs2export['sameagency'] = possiblepairs2export.groupby(['magid', 'patent_id'])['sameagency'].transform('max')
possiblepairs2export['samegrant'] = possiblepairs2export.groupby(['magid', 'patent_id'])['samegrant'].transform('max')
possiblepairs2export = possiblepairs2export.drop_duplicates()
possiblepairs2export = possiblepairs2export.dropna()

In [ ]:
possiblepairs2export.describe()

In [ ]:
# # make sure we have a value for every pair...not necessary in this file
# MISSINGsameagency = 0
# MISSINGsamegrant = 0
# possiblepairs2export = pd.merge(possiblepairs2export, possiblepairs, on=['magid', 'patent_id'], how='right')
# possiblepairs2export['sameagency'] = possiblepairs2export['sameagency'].fillna(MISSINGsameagency)
# possiblepairs2export['samegrant'] = possiblepairs2export['samegrant'].fillna(MISSINGsamegrant)
# possiblepairs2export

In [ ]:
possiblepairs2export.to_parquet(dataint + 'possiblepairs_govoverlap.parquet')